# Overfitting Check

Loads the best tuned model and compares train vs test performance to verify the model is generalizing, not memorizing.

In [ ]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Find project root
current = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [current, *current.parents]:
    if (candidate / 'data' / 'raw').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project root containing data/raw')

# Load data
df = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'ev_features.parquet')

SPLIT_DATE = pd.Timestamp('2022-07-01')
train = df[df['timestamp_hour'] < SPLIT_DATE]
test  = df[df['timestamp_hour'] >= SPLIT_DATE]

EXCLUDE = ['customer_id', 'timestamp_hour', 'target', 'year']
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]

X_train, y_train = train[FEATURE_COLS], train['target']
X_test,  y_test  = test[FEATURE_COLS],  test['target']

# Load saved model
model = joblib.load(PROJECT_ROOT / 'results' / 'best_model.pkl')
print(f'Loaded model: {type(model).__name__}')
print(f'Train: {len(X_train):,} rows | Test: {len(X_test):,} rows')

In [2]:
# --- Predict on both sets ---
y_pred_train = np.clip(model.predict(X_train), 0, None)
y_pred_test  = np.clip(model.predict(X_test), 0, None)

# --- Compute metrics ---
metrics = pd.DataFrame({
    'Set':  ['Train', 'Test'],
    'MAE':  [mean_absolute_error(y_train, y_pred_train), mean_absolute_error(y_test, y_pred_test)],
    'RMSE': [np.sqrt(mean_squared_error(y_train, y_pred_train)), np.sqrt(mean_squared_error(y_test, y_pred_test))],
    'R²':   [r2_score(y_train, y_pred_train), r2_score(y_test, y_pred_test)],
})

print('=== Train vs Test Performance ===')
print(metrics.to_string(index=False))

# --- Verdict ---
gap = metrics.loc[0, 'R²'] - metrics.loc[1, 'R²']
print(f'\nR² gap (train - test): {gap:.4f}')

if gap < 0.03:
    print('→ No overfitting. Model generalizes well.')
elif gap < 0.07:
    print('→ Mild overfitting — acceptable for this dataset size.')
else:
    print('→ Overfitting detected — consider more regularization or fewer features.')

=== Train vs Test Performance ===
  Set      MAE     RMSE       R²
Train 1.467760 3.746965 0.842995
 Test 1.676007 4.399716 0.826766

R² gap (train - test): 0.0162
→ No overfitting. Model generalizes well.
